# Stage 3.3: Prefix Caching for Production Serving

## Problem Statement
Many inference requests share common prefixes (system prompts, conversation history, documents). Recomputing these prefixes wastes compute and increases latency. How can we cache and reuse computation for shared prefixes?

## What You'll Learn
- What is prefix caching and why it matters
- Use cases: Multi-turn conversations, document QA, few-shot learning
- System prompts and shared prefixes
- Implementation with vLLM and other frameworks
- Speedup measurements (2-10x for conversations)
- Cache hit rate analysis
- Production deployment patterns

## Prerequisites
- Understanding of KV cache (notebook 10)
- PagedAttention concepts (notebook 31)
- Basic understanding of caching principles

---

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import hashlib
import time
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
from collections import OrderedDict

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---

## Part 1: The Problem - Redundant Computation

### Common Patterns with Shared Prefixes

#### 1. Multi-turn Conversations
```
Turn 1: [System] You are a helpful assistant.
        [User] Hello
        [AI] Hi! How can I help?

Turn 2: [System] You are a helpful assistant.
        [User] Hello
        [AI] Hi! How can I help?
        [User] What's the weather?
        ^^^^^^^^^^^^^^^^^^^^^^^^^ New content
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ Same as Turn 1 (redundant!)

Turn 3: [System] You are a helpful assistant.
        [User] Hello
        [AI] Hi! How can I help?
        [User] What's the weather?
        [AI] I need your location.
        [User] San Francisco
        ^^^^^^^^^^^^^^^^^^^^ New content
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ Same as Turn 2 (redundant!)
```

**Without prefix caching:** Recompute entire conversation history every turn

**With prefix caching:** Only compute new tokens

#### 2. Document QA
```
Prompt: [Document: 8000 tokens] + [Question: 50 tokens]

Question 1: Document + "What is the main topic?"
Question 2: Document + "Who are the authors?"
Question 3: Document + "When was it published?"

Document (8000 tokens) is the same for all questions!
Without caching: 3 × 8000 = 24,000 tokens processed
With caching: 8000 + 3 × 50 = 8,150 tokens processed (3x speedup!)
```

#### 3. Few-shot Learning
```
Prompt: [Examples: 2000 tokens] + [Query: 100 tokens]

Examples (few-shot demonstrations) are the same for all queries!
With 10 queries: 10 × 2000 = 20,000 tokens without caching
                 2000 + 10 × 100 = 3,000 tokens with caching (6.7x speedup!)
```

In [ ]:
def simulate_inference_cost(num_tokens: int, time_per_token: float = 0.01) -> float:
    """Simulate inference time based on number of tokens"""
    return num_tokens * time_per_token


# Example: Multi-turn conversation
print("=== MULTI-TURN CONVERSATION EXAMPLE ===")
print("\nScenario: 5-turn conversation with system prompt\n")

system_prompt_tokens = 50
turn_lengths = [20, 30, 40, 25, 35]  # New tokens per turn

# Without prefix caching
without_caching_costs = []
cumulative_tokens = system_prompt_tokens

for i, turn_tokens in enumerate(turn_lengths):
    cumulative_tokens += turn_tokens
    cost = simulate_inference_cost(cumulative_tokens)
    without_caching_costs.append(cost)
    print(f"Turn {i+1}: Process {cumulative_tokens} tokens, time = {cost:.3f}s")

total_without = sum(without_caching_costs)
print(f"\nTotal time without caching: {total_without:.3f}s")

# With prefix caching
print("\n" + "="*50)
print("WITH PREFIX CACHING:")
print("="*50 + "\n")

with_caching_costs = []
# First turn: compute system prompt + first user message
first_turn_cost = simulate_inference_cost(system_prompt_tokens + turn_lengths[0])
with_caching_costs.append(first_turn_cost)
print(f"Turn 1: Process {system_prompt_tokens + turn_lengths[0]} tokens (initial), time = {first_turn_cost:.3f}s")

# Subsequent turns: only compute new tokens
for i, turn_tokens in enumerate(turn_lengths[1:], start=2):
    cost = simulate_inference_cost(turn_tokens)
    with_caching_costs.append(cost)
    print(f"Turn {i}: Process {turn_tokens} tokens (new only), time = {cost:.3f}s")

total_with = sum(with_caching_costs)
print(f"\nTotal time with caching: {total_with:.3f}s")

speedup = total_without / total_with
print(f"\n{'='*50}")
print(f"SPEEDUP: {speedup:.2f}x")
print(f"Time saved: {(total_without - total_with):.3f}s ({(1 - total_with/total_without)*100:.1f}%)")
print(f"{'='*50}")

---

## Part 2: How Prefix Caching Works

### Key Concepts

1. **Prefix Identification**
   - Hash the prefix tokens to create a unique identifier
   - Store KV cache for that prefix

2. **Cache Lookup**
   - For new request, hash the prefix
   - Check if prefix exists in cache
   - If hit: Reuse cached KV states
   - If miss: Compute and cache

3. **Cache Management**
   - LRU (Least Recently Used) eviction
   - Reference counting for shared prefixes
   - Cache size limits

### Implementation Architecture

```python
class PrefixCache:
    def __init__(self, max_cache_size: int):
        self.cache = {}  # prefix_hash -> KV_cache
        self.lru = OrderedDict()  # Track access order
        self.max_size = max_cache_size
    
    def get(self, prefix_tokens: List[int]) -> Optional[KVCache]:
        prefix_hash = hash_tokens(prefix_tokens)
        
        if prefix_hash in self.cache:
            # Cache hit - update LRU
            self.lru.move_to_end(prefix_hash)
            return self.cache[prefix_hash]
        
        return None  # Cache miss
    
    def put(self, prefix_tokens: List[int], kv_cache: KVCache):
        prefix_hash = hash_tokens(prefix_tokens)
        
        # Evict if at capacity
        if len(self.cache) >= self.max_size:
            oldest = next(iter(self.lru))
            del self.cache[oldest]
            del self.lru[oldest]
        
        # Add to cache
        self.cache[prefix_hash] = kv_cache
        self.lru[prefix_hash] = True
```

In [ ]:
@dataclass
class KVCache:
    """Simplified KV cache for a sequence"""
    keys: torch.Tensor
    values: torch.Tensor
    num_tokens: int
    
    def memory_size_mb(self) -> float:
        """Estimate memory size in MB"""
        bytes_per_element = 2  # FP16
        total_elements = self.keys.numel() + self.values.numel()
        return (total_elements * bytes_per_element) / (1024 ** 2)


class PrefixCacheManager:
    """
    Production-ready prefix cache manager.
    Simplified version of what vLLM and other frameworks use.
    """
    
    def __init__(self, max_cache_entries: int = 100):
        self.max_cache_entries = max_cache_entries
        
        # Cache storage: prefix_hash -> KVCache
        self.cache: Dict[str, KVCache] = {}
        
        # LRU tracking
        self.lru = OrderedDict()
        
        # Statistics
        self.hits = 0
        self.misses = 0
        self.evictions = 0
    
    def _hash_tokens(self, tokens: List[int]) -> str:
        """Create hash from token sequence"""
        token_bytes = np.array(tokens, dtype=np.int32).tobytes()
        return hashlib.sha256(token_bytes).hexdigest()
    
    def get(self, prefix_tokens: List[int]) -> Optional[KVCache]:
        """Try to retrieve cached KV states for prefix"""
        prefix_hash = self._hash_tokens(prefix_tokens)
        
        if prefix_hash in self.cache:
            # Cache hit
            self.hits += 1
            self.lru.move_to_end(prefix_hash)  # Mark as recently used
            return self.cache[prefix_hash]
        
        # Cache miss
        self.misses += 1
        return None
    
    def put(self, prefix_tokens: List[int], kv_cache: KVCache):
        """Store KV cache for prefix"""
        prefix_hash = self._hash_tokens(prefix_tokens)
        
        # Check if we need to evict
        if len(self.cache) >= self.max_cache_entries and prefix_hash not in self.cache:
            # Evict least recently used
            oldest_hash = next(iter(self.lru))
            del self.cache[oldest_hash]
            del self.lru[oldest_hash]
            self.evictions += 1
        
        # Add to cache
        self.cache[prefix_hash] = kv_cache
        self.lru[prefix_hash] = True
    
    def get_hit_rate(self) -> float:
        """Calculate cache hit rate"""
        total = self.hits + self.misses
        if total == 0:
            return 0.0
        return (self.hits / total) * 100
    
    def get_stats(self) -> Dict:
        """Get cache statistics"""
        total_memory = sum(kv.memory_size_mb() for kv in self.cache.values())
        
        return {
            'cache_size': len(self.cache),
            'max_entries': self.max_cache_entries,
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': self.get_hit_rate(),
            'evictions': self.evictions,
            'memory_mb': total_memory
        }
    
    def clear(self):
        """Clear the cache"""
        self.cache.clear()
        self.lru.clear()


# Demo
print("\n=== PREFIX CACHE MANAGER DEMO ===")

# Create cache manager
cache_manager = PrefixCacheManager(max_cache_entries=10)

# Simulate KV cache for different sequences
def create_dummy_kv_cache(num_tokens: int, num_layers: int = 32, 
                         hidden_dim: int = 4096) -> KVCache:
    """Create a dummy KV cache for demonstration"""
    keys = torch.randn(num_layers, num_tokens, hidden_dim, dtype=torch.float16)
    values = torch.randn(num_layers, num_tokens, hidden_dim, dtype=torch.float16)
    return KVCache(keys, values, num_tokens)

# Example: System prompt
system_prompt = list(range(100))  # 100 tokens
system_kv = create_dummy_kv_cache(100)

print(f"\nCaching system prompt (100 tokens)...")
cache_manager.put(system_prompt, system_kv)
print(f"  Memory used: {system_kv.memory_size_mb():.2f} MB")

# Simulate multiple requests with same system prompt
print("\nSimulating 5 requests with same system prompt:")
for i in range(5):
    # Try to get cached system prompt
    cached_kv = cache_manager.get(system_prompt)
    
    if cached_kv:
        print(f"  Request {i+1}: Cache HIT (saved {cached_kv.num_tokens} tokens)")
    else:
        print(f"  Request {i+1}: Cache MISS (need to compute)")

# Show statistics
stats = cache_manager.get_stats()
print(f"\n=== CACHE STATISTICS ===")
print(f"  Hit rate: {stats['hit_rate']:.1f}%")
print(f"  Hits: {stats['hits']}")
print(f"  Misses: {stats['misses']}")
print(f"  Cache size: {stats['cache_size']}/{stats['max_entries']}")
print(f"  Memory usage: {stats['memory_mb']:.2f} MB")

---

## Part 3: Multi-Turn Conversation Simulation

In [ ]:
def simulate_conversation_without_caching(system_prompt_tokens: int, 
                                         turns: List[int]) -> Tuple[float, List[float]]:
    """
    Simulate conversation without prefix caching.
    Must process entire history every turn.
    """
    time_per_token = 0.01  # 10ms per token
    
    cumulative_tokens = system_prompt_tokens
    turn_times = []
    
    for turn_tokens in turns:
        cumulative_tokens += turn_tokens
        # Process entire conversation history
        turn_time = cumulative_tokens * time_per_token
        turn_times.append(turn_time)
    
    return sum(turn_times), turn_times


def simulate_conversation_with_caching(system_prompt_tokens: int,
                                      turns: List[int]) -> Tuple[float, List[float]]:
    """
    Simulate conversation with prefix caching.
    Only process new tokens after first turn.
    """
    time_per_token = 0.01
    
    turn_times = []
    
    # First turn: process system prompt + first message
    first_turn_time = (system_prompt_tokens + turns[0]) * time_per_token
    turn_times.append(first_turn_time)
    
    # Subsequent turns: only process new tokens
    for turn_tokens in turns[1:]:
        turn_time = turn_tokens * time_per_token
        turn_times.append(turn_time)
    
    return sum(turn_times), turn_times


# Simulate a realistic conversation
system_prompt_tokens = 50
conversation_turns = [30, 40, 35, 50, 45, 40, 55, 35]  # 8 turns

print("\n=== CONVERSATION SIMULATION ===")
print(f"System prompt: {system_prompt_tokens} tokens")
print(f"Number of turns: {len(conversation_turns)}")
print(f"Tokens per turn: {conversation_turns}")

# Without caching
total_without, times_without = simulate_conversation_without_caching(
    system_prompt_tokens, conversation_turns
)

# With caching
total_with, times_with = simulate_conversation_with_caching(
    system_prompt_tokens, conversation_turns
)

print(f"\nWithout caching:")
print(f"  Total time: {total_without:.3f}s")
print(f"  Average per turn: {np.mean(times_without):.3f}s")

print(f"\nWith caching:")
print(f"  Total time: {total_with:.3f}s")
print(f"  Average per turn: {np.mean(times_with):.3f}s")

speedup = total_without / total_with
print(f"\nSpeedup: {speedup:.2f}x")
print(f"Time saved: {total_without - total_with:.3f}s ({(1 - total_with/total_without)*100:.1f}%)")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Per-turn latency comparison
turns = np.arange(1, len(conversation_turns) + 1)
ax1.plot(turns, times_without, marker='o', linewidth=2, markersize=8, 
        label='Without Caching', color='#e74c3c')
ax1.plot(turns, times_with, marker='s', linewidth=2, markersize=8,
        label='With Caching', color='#2ecc71')

ax1.set_xlabel('Turn Number', fontsize=12)
ax1.set_ylabel('Latency (seconds)', fontsize=12)
ax1.set_title('Per-Turn Latency: With vs Without Prefix Caching', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Cumulative time
cumulative_without = np.cumsum(times_without)
cumulative_with = np.cumsum(times_with)

ax2.plot(turns, cumulative_without, marker='o', linewidth=2, markersize=8,
        label='Without Caching', color='#e74c3c')
ax2.plot(turns, cumulative_with, marker='s', linewidth=2, markersize=8,
        label='With Caching', color='#2ecc71')

# Shade the saved time
ax2.fill_between(turns, cumulative_with, cumulative_without, 
                alpha=0.3, color='green', label='Time Saved')

ax2.set_xlabel('Turn Number', fontsize=12)
ax2.set_ylabel('Cumulative Time (seconds)', fontsize=12)
ax2.set_title('Cumulative Time Over Conversation', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('prefix_caching_conversation.png', dpi=300, bbox_inches='tight')
plt.show()

---

## Part 4: Document QA - Extreme Prefix Caching Benefits

In [ ]:
def simulate_document_qa(document_tokens: int, 
                        question_tokens: List[int]) -> Dict:
    """
    Simulate document QA workload with and without prefix caching.
    """
    time_per_token = 0.01
    
    # Without caching: process document for every question
    total_without = sum(
        (document_tokens + q_tokens) * time_per_token 
        for q_tokens in question_tokens
    )
    
    # With caching: process document once, then only questions
    # First question: process document + question
    first_question_time = (document_tokens + question_tokens[0]) * time_per_token
    
    # Subsequent questions: only process question (document cached)
    subsequent_time = sum(q_tokens * time_per_token for q_tokens in question_tokens[1:])
    
    total_with = first_question_time + subsequent_time
    
    return {
        'without_caching': total_without,
        'with_caching': total_with,
        'speedup': total_without / total_with,
        'time_saved': total_without - total_with,
        'tokens_processed_without': sum(document_tokens + q for q in question_tokens),
        'tokens_processed_with': document_tokens + sum(question_tokens)
    }


# Simulate document QA workload
print("\n=== DOCUMENT QA SIMULATION ===")

document_sizes = [1000, 2000, 4000, 8000, 16000]  # Document sizes in tokens
num_questions = 10
question_tokens = [50] * num_questions  # Each question is 50 tokens

results = []

for doc_size in document_sizes:
    result = simulate_document_qa(doc_size, question_tokens)
    results.append(result)
    
    print(f"\nDocument: {doc_size} tokens, {num_questions} questions:")
    print(f"  Without caching: {result['without_caching']:.2f}s")
    print(f"  With caching:    {result['with_caching']:.2f}s")
    print(f"  Speedup:         {result['speedup']:.2f}x")
    print(f"  Token reduction: {result['tokens_processed_without']} → {result['tokens_processed_with']}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Time comparison
times_without = [r['without_caching'] for r in results]
times_with = [r['with_caching'] for r in results]

x = np.arange(len(document_sizes))
width = 0.35

bars1 = ax1.bar(x - width/2, times_without, width, label='Without Caching', 
               color='#e74c3c', alpha=0.8)
bars2 = ax1.bar(x + width/2, times_with, width, label='With Caching',
               color='#2ecc71', alpha=0.8)

ax1.set_xlabel('Document Size (tokens)', fontsize=12)
ax1.set_ylabel('Total Time (seconds)', fontsize=12)
ax1.set_title(f'Document QA: {num_questions} Questions', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(document_sizes)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

# Speedup
speedups = [r['speedup'] for r in results]

ax2.plot(document_sizes, speedups, marker='o', linewidth=2, markersize=10,
        color='#9b59b6')
ax2.fill_between(document_sizes, 1, speedups, alpha=0.3, color='#9b59b6')
ax2.axhline(y=1, color='gray', linestyle='--', linewidth=1, label='No speedup')

for i, speedup in enumerate(speedups):
    ax2.text(document_sizes[i], speedup, f'{speedup:.1f}x',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.set_xlabel('Document Size (tokens)', fontsize=12)
ax2.set_ylabel('Speedup', fontsize=12)
ax2.set_title('Prefix Caching Speedup vs Document Size', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('prefix_caching_document_qa.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== KEY INSIGHT ===")
print("Speedup increases with document size!")
print(f"At {document_sizes[-1]} tokens: {speedups[-1]:.1f}x speedup")

---

## Part 5: Cache Hit Rate Analysis

In [ ]:
def simulate_cache_hit_rates(num_system_prompts: int,
                            num_requests: int,
                            request_distribution: str = 'uniform') -> Dict:
    """
    Simulate cache hit rates under different scenarios.
    """
    cache_manager = PrefixCacheManager(max_cache_entries=num_system_prompts)
    
    # Create system prompts
    system_prompts = [
        list(range(i * 100, (i + 1) * 100)) 
        for i in range(num_system_prompts)
    ]
    
    # Generate request pattern
    if request_distribution == 'uniform':
        # Uniform: each system prompt equally likely
        request_prompts = np.random.choice(
            range(num_system_prompts), 
            size=num_requests
        )
    elif request_distribution == 'skewed':
        # Skewed: 80% of requests use top 20% of prompts (Pareto)
        top_prompts = int(num_system_prompts * 0.2)
        request_prompts = []
        for _ in range(num_requests):
            if np.random.random() < 0.8:
                # Use top 20%
                request_prompts.append(np.random.choice(range(top_prompts)))
            else:
                # Use bottom 80%
                request_prompts.append(np.random.choice(range(top_prompts, num_system_prompts)))
        request_prompts = np.array(request_prompts)
    
    # Process requests
    for prompt_idx in request_prompts:
        prompt = system_prompts[prompt_idx]
        
        # Try to get from cache
        cached = cache_manager.get(prompt)
        
        if not cached:
            # Cache miss - compute and store
            kv_cache = create_dummy_kv_cache(100)
            cache_manager.put(prompt, kv_cache)
    
    return cache_manager.get_stats()


# Analyze cache hit rates
print("\n=== CACHE HIT RATE ANALYSIS ===")

num_prompts = 50
num_requests = 1000

print(f"\nScenario: {num_prompts} different system prompts, {num_requests} requests\n")

# Uniform distribution
uniform_stats = simulate_cache_hit_rates(num_prompts, num_requests, 'uniform')
print("Uniform distribution (all prompts equally likely):")
print(f"  Hit rate: {uniform_stats['hit_rate']:.1f}%")
print(f"  Hits: {uniform_stats['hits']}")
print(f"  Misses: {uniform_stats['misses']}")
print(f"  Evictions: {uniform_stats['evictions']}")

# Skewed distribution (realistic)
skewed_stats = simulate_cache_hit_rates(num_prompts, num_requests, 'skewed')
print("\nSkewed distribution (80/20 rule):")
print(f"  Hit rate: {skewed_stats['hit_rate']:.1f}%")
print(f"  Hits: {skewed_stats['hits']}")
print(f"  Misses: {skewed_stats['misses']}")
print(f"  Evictions: {skewed_stats['evictions']}")

print("\n=== INSIGHT ===")
print("Real-world workloads are typically skewed (80/20 rule)")
print("Most requests use a small subset of system prompts")
print(f"Hit rate: {skewed_stats['hit_rate']:.1f}% (much better than uniform!)")

# Visualize hit rate vs cache size
cache_sizes = [10, 20, 30, 40, 50, 75, 100]
hit_rates_uniform = []
hit_rates_skewed = []

for cache_size in cache_sizes:
    uniform = simulate_cache_hit_rates(cache_size, 1000, 'uniform')
    skewed = simulate_cache_hit_rates(cache_size, 1000, 'skewed')
    hit_rates_uniform.append(uniform['hit_rate'])
    hit_rates_skewed.append(skewed['hit_rate'])

plt.figure(figsize=(10, 6))
plt.plot(cache_sizes, hit_rates_uniform, marker='o', linewidth=2, markersize=8,
        label='Uniform Distribution', color='#3498db')
plt.plot(cache_sizes, hit_rates_skewed, marker='s', linewidth=2, markersize=8,
        label='Skewed Distribution (80/20)', color='#2ecc71')

plt.xlabel('Cache Size (number of prompts)', fontsize=12)
plt.ylabel('Hit Rate (%)', fontsize=12)
plt.title('Cache Hit Rate vs Cache Size', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(0, 105)

plt.tight_layout()
plt.savefig('cache_hit_rate_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

---

## Part 6: Production Implementation with vLLM

### Using vLLM's Automatic Prefix Caching

vLLM automatically handles prefix caching when enabled:

```python
from vllm import LLM, SamplingParams

# Initialize vLLM with prefix caching enabled
llm = LLM(
    model="meta-llama/Llama-2-7b-hf",
    enable_prefix_caching=True,  # Enable automatic prefix caching
    gpu_memory_utilization=0.9
)

# System prompt (will be cached)
system_prompt = "You are a helpful AI assistant."

# Multi-turn conversation
conversation_history = []

for user_input in user_messages:
    # Build full prompt with history
    prompt = system_prompt + "\n" + "\n".join(conversation_history) + "\n" + user_input
    
    # vLLM automatically detects shared prefix and caches it
    outputs = llm.generate(prompt, sampling_params)
    
    # Add to history
    conversation_history.append(f"User: {user_input}")
    conversation_history.append(f"Assistant: {outputs[0].outputs[0].text}")
```

### Key Features

1. **Automatic Detection**: vLLM automatically detects shared prefixes
2. **Hash-based Matching**: Uses token hashing for fast lookups
3. **LRU Eviction**: Automatically manages cache size
4. **PagedAttention Integration**: Works seamlessly with PagedAttention

### Configuration Options

```python
llm = LLM(
    model="meta-llama/Llama-2-7b-hf",
    enable_prefix_caching=True,
    
    # Cache configuration
    prefix_cache_size=1000,  # Max number of cached prefixes
    
    # Memory management
    gpu_memory_utilization=0.9,
    max_num_seqs=256,  # Max concurrent sequences
)
```

---

## Part 7: Production Patterns & Best Practices

### Pattern 1: Multi-Turn Conversations

```python
class ConversationManager:
    def __init__(self, llm, system_prompt: str):
        self.llm = llm
        self.system_prompt = system_prompt
        self.conversations = {}  # session_id -> history
    
    def add_turn(self, session_id: str, user_message: str) -> str:
        # Get or create conversation history
        if session_id not in self.conversations:
            self.conversations[session_id] = []
        
        history = self.conversations[session_id]
        
        # Build prompt with shared system prompt
        prompt = self.system_prompt + "\n"
        prompt += "\n".join(history)
        prompt += f"\nUser: {user_message}\nAssistant:"
        
        # Generate (prefix will be cached automatically)
        response = self.llm.generate(prompt)
        
        # Update history
        history.append(f"User: {user_message}")
        history.append(f"Assistant: {response}")
        
        return response
```

### Pattern 2: Document QA Service

```python
class DocumentQAService:
    def __init__(self, llm):
        self.llm = llm
        self.document_cache = {}  # doc_id -> document_text
    
    def upload_document(self, doc_id: str, document: str):
        # Store document (will be used as prefix)
        self.document_cache[doc_id] = document
    
    def ask_question(self, doc_id: str, question: str) -> str:
        if doc_id not in self.document_cache:
            raise ValueError("Document not found")
        
        # Build prompt with document as prefix
        document = self.document_cache[doc_id]
        prompt = f"Document:\n{document}\n\nQuestion: {question}\nAnswer:"
        
        # Document prefix will be cached after first question
        response = self.llm.generate(prompt)
        
        return response
```

### Pattern 3: Few-Shot Learning

```python
class FewShotClassifier:
    def __init__(self, llm, examples: List[Tuple[str, str]]):
        self.llm = llm
        
        # Build few-shot prefix (will be shared across all queries)
        self.few_shot_prefix = "Classify the following text:\n\n"
        for text, label in examples:
            self.few_shot_prefix += f"Text: {text}\nLabel: {label}\n\n"
    
    def classify(self, text: str) -> str:
        # Append query to cached prefix
        prompt = self.few_shot_prefix + f"Text: {text}\nLabel:"
        
        # Few-shot examples cached, only process new text
        response = self.llm.generate(prompt)
        
        return response
```

### Best Practices

1. **Consistent Prefixes**: Keep system prompts identical for maximum cache hits
2. **Prefix Length**: Longer shared prefixes = bigger wins
3. **Cache Warming**: Pre-populate cache with common prompts
4. **Monitor Hit Rates**: Track cache effectiveness in production
5. **Memory Management**: Balance cache size with memory constraints

---

## Part 8: Key Takeaways

### Why Prefix Caching Matters

1. **Massive Speedups for Common Patterns**
   - Multi-turn conversations: 2-10x faster
   - Document QA: 3-15x faster (depending on document size)
   - Few-shot learning: 5-20x faster

2. **Cost Reduction**
   - Process fewer tokens = lower compute costs
   - Higher throughput = better hardware utilization

3. **Better User Experience**
   - Lower latency for subsequent turns
   - More responsive chatbots
   - Faster document analysis

### When Prefix Caching Helps Most

| Use Case | Shared Prefix | Speedup |
|----------|---------------|----------|
| Multi-turn chat | System prompt + history | 2-10x |
| Document QA | Document context | 3-15x |
| Few-shot learning | Example demonstrations | 5-20x |
| Batch processing | Common instructions | 2-5x |
| Code generation | Codebase context | 3-10x |

### Production Metrics to Track

```python
# Monitor these in production:
metrics = {
    'cache_hit_rate': 0.85,  # Target: >80%
    'avg_prefix_length': 500,  # Longer = better
    'cache_memory_mb': 2048,  # Monitor memory usage
    'speedup': 6.5,  # Actual speedup observed
    'tokens_saved': 1_000_000,  # Cumulative tokens not recomputed
}
```

### Implementation Checklist

- [ ] Enable prefix caching in vLLM/serving framework
- [ ] Design prompts with consistent structure
- [ ] Keep system prompts identical across requests
- [ ] Monitor cache hit rates
- [ ] Adjust cache size based on workload
- [ ] Pre-warm cache with common prefixes
- [ ] Track cost savings from caching

---

## References & Next Steps

### Related Notebooks
- **Notebook 10**: KV cache fundamentals
- **Notebook 30**: Continuous batching (works well with prefix caching)
- **Notebook 31**: PagedAttention (enables efficient prefix sharing)
- **Notebook 34**: Multi-GPU serving

### Framework Documentation
- **vLLM**: https://docs.vllm.ai/en/latest/models/engine_args.html#prefix-caching
- **SGLang**: Advanced prefix caching implementation
- **TensorRT-LLM**: Supports prefix caching

### Papers
- "Efficient Memory Management for Large Language Model Serving with PagedAttention" (vLLM paper)
- SGLang paper on RadixAttention (advanced prefix caching)

### LLM Inference Roadmap
- See `LLM_INFERENCE_ROADMAP.md` - Stage 3: Advanced Serving

---

## Summary

Prefix caching is a **critical optimization** for production LLM serving:

1. **Identifies shared prefixes** across requests (system prompts, documents, examples)
2. **Caches KV states** for shared prefixes
3. **Reuses cached computation** for subsequent requests
4. **Delivers 2-20x speedups** depending on use case

**Key principle:** Don't recompute what you've already computed.

Combined with continuous batching and PagedAttention, prefix caching enables highly efficient production serving.

**Next:** Notebook 34 - Multi-GPU Serving (scaling across multiple GPUs for large models)